# Nemotron v62 — D3 Sparse-Trust Finisher Attack

Safe follow-up to v61. v61 used clean D3 substitution-cipher traces but applied the tiny finisher delta to 8k+ tensors and stayed neutral at 0.86. v62 keeps the same fast D3 generation and short finisher SFT, but changes the final merge to a sparse-trust blend:

- Huikang/BorgQueen adapter is preferred when attached; Kien/v14 is fallback.
- D4 stays disabled by default to avoid the v60 hang.
- The finisher delta is applied only to the top stable tensors, not broadly.
- Default sparse target: 600 tensors.
- `submission.zip` is created only after the real v62 candidate succeeds.

Submit only if the final output says `FINAL v62_d3_sparse_trust_finisher_candidate`.


In [ ]:

# CELL 1 — Configuration
import os, sys, json, glob, shutil, zipfile, re, gc, math, random, time, subprocess, inspect, traceback, hashlib
from pathlib import Path
from collections import Counter, defaultdict

os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"): sys.stderr.reconfigure(encoding="utf-8", errors="replace")

SEED = 62
random.seed(SEED)

WORK = Path("/kaggle/working"); WORK.mkdir(parents=True, exist_ok=True)
TMP_ROOT = Path("/tmp/nemotron_v62_work")
if TMP_ROOT.exists(): shutil.rmtree(TMP_ROOT, ignore_errors=True)
TMP_ROOT.mkdir(parents=True, exist_ok=True)
OUT_DIR = WORK / "v62_outputs"
SUBMISSION_ZIP = WORK / "submission.zip"
FALLBACK_ZIP = TMP_ROOT / "FALLBACK_DO_NOT_SUBMIT_base.zip"

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
BASE_MODEL_PATH = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")
COMP_TRAIN_PATH = Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv")

# Edit this to force the adapter that you know scored 0.86.
START_ADAPTER_PATH = ""
AUTO_ADAPTER_CANDIDATES = [
    # Prefer Huikang / BorgQueen / GlyphMatics lane if attached.
    "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/huikang/nemotron-adapter/transformers/default/20",
    "/kaggle/input/nemotron-adapter/transformers/default/20",
    # Kien/v14 fallback.
    "/kaggle/input/models/kienngx/nemotron-nano-30b-trained/triton/tinker-adapter/1",
]

SUB_REPAIR_MAX_ROWS = 260
# D4 was the v60 hang point. Keep OFF by default; if enabled, hard-cap it.
ENABLE_D4 = False
CRYPTO_REPAIR_MAX_ROWS = 80
CRYPTO_SCAN_MAX_ROWS = 180
CRYPTO_ROW_TIMEOUT_SEC = 0.15

D3_UPSAMPLE = 5
D4_UPSAMPLE = 2
PROTECTIVE_PER_FAMILY = {
    "bit_manipulation": 220,
    "cipher": 120,
    "equation_numeric": 160,
    "numeral": 180,
    "gravity": 180,
    "unit_conversion": 180,
}
MAX_TRAIN_ROWS = 3000

MAX_SEQ_LEN = 4096
PER_DEVICE_BS = 1
GRAD_ACCUM = 16
MAX_STEPS = 14
LR = 9e-8
WARMUP_STEPS = 2

BLEND_ALPHA = 0.0010
PRESERVE_LM_HEAD = True
SKIP_LATE_EXPERTS = True
LATE_EXPERT_LAYER_START = 38
MAX_REL_DELTA = 0.0022
MAX_ABSMAX_DELTA = 0.0060
# Sparse-trust merge: v61 applied ~8351 tensors. v62 keeps only a small, high-confidence slice.
MAX_SELECTED_TENSORS = 600
MIN_SELECTED_TENSORS = 250
EXPERT_LAYER_HARD_CAP = 30
EXPERT_MAX_REL_DELTA = 0.0016
EXPERT_MAX_FRACTION = 0.55
NONEXPERT_MIN_REL_DELTA = 1e-7
REQUIRE_MIN_APPLIED = MIN_SELECTED_TENSORS

TRAIN_DIR = TMP_ROOT / "sft_v62"
TRAINED_ADAPTER_DIR = TMP_ROOT / "adapter_v62_trained"
BLENDED_ADAPTER_DIR = TMP_ROOT / "adapter_v62_blended"

print({
    "version": "v62_d3_sparse_trust_finisher",
    "MAX_STEPS": MAX_STEPS, "LR": LR, "BLEND_ALPHA": BLEND_ALPHA, "MAX_SELECTED_TENSORS": MAX_SELECTED_TENSORS,
    "D3_UPSAMPLE": D3_UPSAMPLE, "ENABLE_D4": ENABLE_D4, "D4_UPSAMPLE": D4_UPSAMPLE, "CRYPTO_ROW_TIMEOUT_SEC": CRYPTO_ROW_TIMEOUT_SEC,
    "START_ADAPTER_PATH": START_ADAPTER_PATH or "AUTO",
})


In [ ]:

# CELL 2 — Disk, adapter detection, and packaging helpers
def show_disk(label=""):
    print(f"\n--- DISK {label} ---")
    os.system("df -h /kaggle/working /tmp 2>/dev/null || true")
    os.system("du -sh /kaggle/working/* 2>/dev/null | sort -h | tail -20 || true")
    os.system("du -sh /tmp/nemotron_v62_work/* 2>/dev/null | sort -h | tail -20 || true")

def rm_path(p):
    p = Path(p)
    if p.exists():
        if p.is_dir(): shutil.rmtree(p, ignore_errors=True)
        else: p.unlink(missing_ok=True)

def free_gb(path="/kaggle/working"):
    return shutil.disk_usage(path).free / (1024**3)

show_disk("start")
for pat in ["submission*.zip", "tmp_*", "adapter_v60*", "adapter_v61*", "adapter_v62*", "sft_v60*", "sft_v61*", "sft_v62*", "v60_*", "v61_*", "v62_*", "pydeps"]:
    for p in WORK.glob(pat): rm_path(p)
OUT_DIR.mkdir(parents=True, exist_ok=True)
if SUBMISSION_ZIP.exists(): SUBMISSION_ZIP.unlink()
gc.collect()
show_disk("after cleanup")

assert BASE_MODEL_PATH.exists(), BASE_MODEL_PATH
assert COMP_TRAIN_PATH.exists(), COMP_TRAIN_PATH
assert free_gb("/kaggle/working") > 7.5

def find_adapter_path():
    if START_ADAPTER_PATH and Path(START_ADAPTER_PATH).exists():
        return Path(START_ADAPTER_PATH)
    for p in AUTO_ADAPTER_CANDIDATES:
        q = Path(p)
        if (q / "adapter_config.json").exists() and (q / "adapter_model.safetensors").exists():
            return q
    hits = []
    for cfg in Path("/kaggle/input").rglob("adapter_config.json"):
        folder = cfg.parent
        if not (folder / "adapter_model.safetensors").exists(): continue
        s = str(folder).lower()
        score = 0
        # Strongly prefer known 0.86 Huikang/Borg/Glyph lane if present.
        for tok, w in [("huikang",50),("borg",45),("glyph",35),("nemotron-adapter",30),("kien",10),("tinker",8),("adapter",5)]:
            if tok in s: score += w
        score -= len(str(folder)) / 1000
        hits.append((score, folder))
    hits.sort(reverse=True, key=lambda x: x[0])
    if hits: return hits[0][1]
    raise FileNotFoundError("Could not find starting adapter.")

START_ADAPTER = find_adapter_path()
print("START_ADAPTER:", START_ADAPTER)
assert (START_ADAPTER / "adapter_config.json").exists()
assert (START_ADAPTER / "adapter_model.safetensors").exists()

def validate_submission_zip(zip_path, min_zip_gb=2.0, min_weight_gb=2.0):
    zip_path = Path(zip_path); assert zip_path.exists(), f"Missing {zip_path}"
    size = zip_path.stat().st_size / 1024**3
    assert size > min_zip_gb, f"Zip too small {size:.4f}GB"
    with zipfile.ZipFile(zip_path, "r") as z:
        names = z.namelist()
        assert set(names) == {"adapter_config.json", "adapter_model.safetensors"}, names
        for n in names:
            assert "/" not in n.strip("/"), f"Nested file {n}"
            info = z.getinfo(n); assert info.file_size > 0
            if n == "adapter_model.safetensors":
                assert info.file_size / 1024**3 > min_weight_gb
    return size

def patch_config_for_submission(src_cfg):
    cfg = dict(src_cfg)
    cfg["base_model_name_or_path"] = BASE_MODEL_NAME
    cfg["inference_mode"] = True
    if "lora_dropout" in cfg: cfg["lora_dropout"] = 0.0
    assert cfg.get("peft_type") == "LORA"
    assert int(cfg.get("r", 999)) <= 32
    return cfg

def package_adapter_to_zip(adapter_dir, out_zip):
    adapter_dir = Path(adapter_dir); out_zip = Path(out_zip)
    assert (adapter_dir / "adapter_config.json").exists()
    assert (adapter_dir / "adapter_model.safetensors").exists()
    tmp_cfg = TMP_ROOT / f"tmp_{out_zip.stem}_adapter_config.json"
    tmp_zip = TMP_ROOT / f"tmp_{out_zip.stem}.zip"
    tmp_cfg.unlink(missing_ok=True); tmp_zip.unlink(missing_ok=True)
    cfg = patch_config_for_submission(json.load(open(adapter_dir / "adapter_config.json", "r", encoding="utf-8")))
    tmp_cfg.write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    with zipfile.ZipFile(tmp_zip, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as z:
        z.write(tmp_cfg, "adapter_config.json")
        z.write(adapter_dir / "adapter_model.safetensors", "adapter_model.safetensors")
    tmp_cfg.unlink(missing_ok=True)
    validate_submission_zip(tmp_zip)
    if out_zip.exists(): out_zip.unlink()
    shutil.move(str(tmp_zip), str(out_zip))
    return validate_submission_zip(out_zip)

package_adapter_to_zip(START_ADAPTER, FALLBACK_ZIP)
print("Created fallback DO-NOT-SUBMIT:", FALLBACK_ZIP, validate_submission_zip(FALLBACK_ZIP))
show_disk("after fallback diagnostic")


In [ ]:

# CELL 3 — Utilities and train.csv family classifier
import pandas as pd
import numpy as np

_BOX_RE = re.compile(r"\\boxed\s*\{")
def strip_control_chars(s):
    return "".join(ch for ch in str(s) if (ord(ch) >= 32 or ch in "\n\t\r"))

def extract_boxed_all(text):
    text = str(text or ""); boxes = []
    for m in _BOX_RE.finditer(text):
        i = m.end(); depth = 1; buf = []
        while i < len(text) and depth > 0:
            c = text[i]
            if c == "\\" and i + 1 < len(text):
                buf.append(c); buf.append(text[i+1]); i += 2; continue
            if c == "{": depth += 1; buf.append(c)
            elif c == "}":
                depth -= 1
                if depth == 0: break
                buf.append(c)
            else: buf.append(c)
            i += 1
        if depth == 0: boxes.append("".join(buf).strip())
    return boxes

def norm_answer(x):
    s = strip_control_chars("" if x is None else str(x).strip())
    s = s.replace("$", "").replace("\\ ", " ").replace("\\,", " ")
    s = re.sub(r"\\text\s*\{([^{}]*)\}", r"\1", s)
    s = re.sub(r"\\mathrm\s*\{([^{}]*)\}", r"\1", s)
    s = re.sub(r"\\boxed\s*\{([^{}]*)\}", r"\1", s)
    s = s.replace("{", "").replace("}", "").replace("\\", "")
    s = re.sub(r"\bmeters?\b|\bm\b", "", s, flags=re.I)
    s = re.sub(r"\s+", " ", s).strip().lower()
    return s.replace("−", "-").replace("–", "-")

def answer_equal(pred, gold):
    if norm_answer(pred) == norm_answer(gold): return True
    try:
        aa = float(re.findall(r"-?\d+(?:\.\d+)?", str(pred))[0])
        bb = float(re.findall(r"-?\d+(?:\.\d+)?", str(gold))[0])
        return abs(aa - bb) <= 0.015
    except Exception:
        return False

def escape_box_answer(ans):
    return str(ans).strip().replace("\\", "\\\\").replace("{", "\\{").replace("}", "\\}")

def family_from_prompt(prompt):
    s = str(prompt or "").lower()
    if "secret bit manipulation rule transforms 8-bit binary numbers" in s or "bit manipulation" in s: return "bit_manipulation"
    if "secret encryption rules are used on text" in s or "decrypt the following text" in s: return "cipher"
    if "secret set of transformation rules is applied to equations" in s: return "equation_numeric"
    if "numbers are secretly converted into a different numeral system" in s or "wonderland numeral" in s: return "numeral"
    if "gravitational constant has been secretly changed" in s or "gravitational" in s: return "gravity"
    if "secret unit conversion is applied to measurements" in s or "unit conversion" in s: return "unit_conversion"
    return "unknown"

def make_chat_target(prompt, assistant):
    return {"messages": [{"role": "user", "content": str(prompt)}, {"role": "assistant", "content": str(assistant)}]}

def answer_only_assistant(answer):
    return f"Apply the demonstrated rule carefully.\n</think>\n\\boxed{{{escape_box_answer(answer)}}}"

comp = pd.read_csv(COMP_TRAIN_PATH)
comp["id"] = comp["id"].astype(str)
comp["family"] = comp["prompt"].map(family_from_prompt)
print("train shape:", comp.shape)
print(comp["family"].value_counts())
assert "unknown" not in set(comp["family"].unique()), comp["family"].value_counts()


In [ ]:

# CELL 4 — D3 substitution-cipher verified traces
SUB_MARKER = "secret encryption rules are used on text"

def is_substitution_prompt(prompt):
    return isinstance(prompt, str) and SUB_MARKER in prompt

def parse_sub_prompt(prompt):
    m = re.search(r"Here are some examples:\s*(.*?)\s*Now,\s*decrypt the following text:\s*(.+?)\s*$", prompt, flags=re.S | re.I)
    if not m: raise ValueError("Could not parse substitution prompt")
    examples = []
    for line in m.group(1).strip().splitlines():
        if "->" not in line: continue
        cipher, plain = line.split("->", 1)
        cipher, plain = cipher.strip().lower(), plain.strip().lower()
        if cipher and plain: examples.append((cipher, plain))
    target = m.group(2).strip().lower()
    return examples, target

def build_vocab_from_substitution_train(df):
    vocab = set()
    for prompt in df["prompt"].astype(str):
        if not is_substitution_prompt(prompt): continue
        try: examples, _ = parse_sub_prompt(prompt)
        except Exception: continue
        for _, plain in examples:
            for w in plain.split():
                if re.fullmatch(r"[a-z]+", w): vocab.add(w)
    return sorted(vocab)

def word_pattern(word):
    seen, out, nxt = {}, [], 0
    for ch in word:
        if ch not in seen: seen[ch] = nxt; nxt += 1
        out.append(seen[ch])
    return tuple(out)

SUB_VOCAB = build_vocab_from_substitution_train(comp)
VOCAB_BY_PATTERN = defaultdict(list)
for w in SUB_VOCAB:
    VOCAB_BY_PATTERN[(len(w), word_pattern(w))].append(w)
print("SubCipher vocab words:", len(SUB_VOCAB))

def mapping_from_examples(examples):
    c2p, p2c = {}, {}
    for cipher, plain in examples:
        if len(cipher) != len(plain): return None, None
        for c, p in zip(cipher, plain):
            if c == " " or p == " ":
                if c != p: return None, None
                continue
            if not (c.isalpha() and p.isalpha()): continue
            if c in c2p and c2p[c] != p: return None, None
            if p in p2c and p2c[p] != c: return None, None
            c2p[c] = p; p2c[p] = c
    return c2p, p2c

def decode_with_map(text, c2p):
    return "".join(c2p.get(ch, "?") if ch.isalpha() else ch for ch in text)

def solve_substitution_prompt(prompt):
    examples, target = parse_sub_prompt(prompt)
    c2p, p2c = mapping_from_examples(examples)
    if c2p is None: return None, {}
    direct = decode_with_map(target, c2p)
    if "?" not in direct: return direct, {"mode": "direct_map", "map": c2p}
    # Light vocabulary CSP fallback.
    target_words = target.split(); decoded_words = direct.split()
    if len(target_words) != len(decoded_words): return None, {}
    c2p0, p2c0 = dict(c2p), dict(p2c)
    candidates_per_word = []
    for cw, dw in zip(target_words, decoded_words):
        if "?" not in dw: candidates_per_word.append([dw]); continue
        pat = (len(cw), word_pattern(cw))
        cands = VOCAB_BY_PATTERN.get(pat, [])
        ok_cands = []
        for w in cands:
            ok = True
            for c, p in zip(cw, w):
                if c in c2p0 and c2p0[c] != p: ok = False; break
                if p in p2c0 and p2c0[p] != c: ok = False; break
            if ok: ok_cands.append(w)
        candidates_per_word.append(ok_cands[:40])
    order = sorted(range(len(candidates_per_word)), key=lambda i: len(candidates_per_word[i]))
    best = []
    def backtrack(pos, c2p_cur, p2c_cur, chosen):
        if len(best) > 1: return
        if pos == len(order):
            ans = [None] * len(target_words)
            for i, w in chosen.items(): ans[i] = w
            best.append(" ".join(ans)); return
        i = order[pos]; cw = target_words[i]
        for plain_w in candidates_per_word[i]:
            c2p_new, p2c_new = dict(c2p_cur), dict(p2c_cur)
            ok = True
            for c, p in zip(cw, plain_w):
                if c in c2p_new and c2p_new[c] != p: ok = False; break
                if p in p2c_new and p2c_new[p] != c: ok = False; break
                c2p_new[c] = p; p2c_new[p] = c
            if ok:
                chosen[i] = plain_w
                backtrack(pos+1, c2p_new, p2c_new, chosen)
                chosen.pop(i, None)
    backtrack(0, c2p0, p2c0, {})
    if len(best) == 1: return best[0], {"mode": "vocab_csp", "map": c2p}
    return None, {"mode": "abstain"}

def make_sub_trace(prompt, pred, info):
    examples, target = parse_sub_prompt(prompt)
    mapping_text = ", ".join(f"{c}->{p}" for c, p in sorted(info.get("map", {}).items())[:40])
    return f"Build the monoalphabetic substitution map from the examples: {mapping_text}. Apply it to \"{target}\" to decode the plaintext as \"{pred}\".\n</think>\n\\boxed{{{escape_box_answer(pred)}}}"

d3_rows, d3_stats = [], Counter()
for _, row in comp[comp["family"] == "cipher"].iterrows():
    try:
        pred, info = solve_substitution_prompt(row["prompt"])
        if pred is None: d3_stats["abstain"] += 1; continue
        if answer_equal(pred, row["answer"]):
            d3_rows.append({"id": row["id"], "family": "cipher", "source": "d3_substitution_solver_verified",
                            "prompt": row["prompt"], "answer": row["answer"], "assistant": make_sub_trace(row["prompt"], pred, info)})
            d3_stats["verified_exact"] += 1
        else: d3_stats["wrong"] += 1
    except Exception: d3_stats["error"] += 1
    if len(d3_rows) >= SUB_REPAIR_MAX_ROWS: break

print("D3 stats:", d3_stats, "rows:", len(d3_rows))
assert len(d3_rows) >= 80, f"Too few D3 verified rows: {len(d3_rows)}"
pd.DataFrame(d3_rows).to_json(OUT_DIR/"v62_d3_substitution_verified.jsonl", orient="records", lines=True, force_ascii=False)


In [ ]:

# CELL 5 — Optional D4 cryptarithm/equation verified traces, hard-capped
# v60 hung here. v61 defaults ENABLE_D4=False. If you set ENABLE_D4=True,
# this cell enforces CRYPTO_SCAN_MAX_ROWS and CRYPTO_ROW_TIMEOUT_SEC.
import signal
CRYPTO_MARKER = "secret set of transformation rules is applied to equations"
_EQ_RE = re.compile(r"^(\S{5})\s*=\s*(\S{1,4})\s*$")
_QUESTION_RE = re.compile(r"Now,\s+determine\s+the\s+result\s+for:\s*(\S{5})\s*$", flags=re.I)
OP_NAMES = ["add", "abs_diff", "mul", "concat", "rev_concat"]

class TimeoutException(Exception): pass

def _timeout_handler(signum, frame):
    raise TimeoutException("D4 row timeout")

def parse_crypto_prompt(prompt):
    lines = [l.strip() for l in str(prompt).splitlines() if l.strip()]
    examples, question = [], None
    for l in lines:
        m = _EQ_RE.match(l)
        if m: examples.append({"input_value": m.group(1), "output_value": m.group(2)})
        q = _QUESTION_RE.search(l)
        if q: question = q.group(1)
    if question is None:
        m = re.search(r"Now,\s+determine\s+the\s+result\s+for:\s*(\S{5})", str(prompt), flags=re.I|re.S)
        if m: question = m.group(1)
    if not examples or question is None: raise ValueError("Could not parse crypto prompt")
    return {"examples": examples, "question": question}

def result_digits_for_op(op_id, left, right):
    if op_id == 0: val = left + right
    elif op_id == 1: val = abs(left - right)
    elif op_id == 2: val = left * right
    elif op_id == 3: val = left * 100 + right
    elif op_id == 4: val = right * 100 + left
    else: return None
    if val < 0 or val > 9999: return None
    return tuple(int(ch) for ch in str(val))

class FastCryptoSolver:
    """Same small symbolic solver idea as v60, but with max-node guards."""
    def __init__(self, parsed, max_solutions=2, max_nodes=25000):
        self.examples=[]
        for ex in parsed["examples"][:6]:
            inp,out=ex["input_value"],ex["output_value"]
            if len(inp)==5 and 1 <= len(out) <= 4:
                self.examples.append((inp[0],inp[1],inp[2],inp[3],inp[4],tuple(out)))
        self.query=parsed["question"]; self.max_solutions=max_solutions; self.max_nodes=max_nodes
        self.nodes=0; self.answers=Counter(); self.answer_info={}; self.mapping={}; self.rev={}; self.op_assign={}
    def vals(self,sym): return [self.mapping[sym]] if sym in self.mapping else list(range(10))
    def assign(self,sym,val):
        if sym in self.mapping: return self.mapping[sym]==val
        if val in self.rev: return False
        self.mapping[sym]=val; self.rev[val]=sym; return True
    def undo(self,sym):
        if sym in self.mapping:
            val=self.mapping.pop(sym); self.rev.pop(val,None)
    def solve(self):
        self._process(0)
        if not self.answers: return None, ({},{})
        pred,cnt=self.answers.most_common(1)[0]
        if len(self.answers)>1 and self.answers.most_common(2)[1][1]==cnt: return None, ({},{})
        return pred, self.answer_info.get(pred,({},{"nodes":self.nodes}))
    def _process(self,idx):
        self.nodes += 1
        if self.nodes > self.max_nodes: return
        if len(self.answers)>=self.max_solutions: return
        if idx>=len(self.examples): self._compute_query(); return
        s0,s1,op_sym,s3,s4,out_syms=self.examples[idx]
        op_options=[self.op_assign[op_sym]] if op_sym in self.op_assign else list(range(len(OP_NAMES)))
        for d0 in self.vals(s0):
            if not self.assign(s0,d0): continue
            for d1 in self.vals(s1):
                if not self.assign(s1,d1): continue
                left=d0*10+d1
                for d3 in self.vals(s3):
                    if not self.assign(s3,d3): continue
                    for d4 in self.vals(s4):
                        if not self.assign(s4,d4): continue
                        right=d3*10+d4
                        for op_id in op_options:
                            rd=result_digits_for_op(op_id,left,right)
                            if rd is None or len(rd)!=len(out_syms): continue
                            assigned=[]; ok=True; op_new=op_sym not in self.op_assign
                            if op_new: self.op_assign[op_sym]=op_id
                            for rsym,rdig in zip(out_syms,rd):
                                if rsym.isdigit():
                                    if int(rsym)!=rdig: ok=False; break
                                else:
                                    if not self.assign(rsym,rdig): ok=False; break
                                    assigned.append(rsym)
                            if ok: self._process(idx+1)
                            for rsym in reversed(assigned): self.undo(rsym)
                            if op_new: self.op_assign.pop(op_sym,None)
                        self.undo(s4)
                    self.undo(s3)
                self.undo(s1)
            self.undo(s0)
    def _compute_query(self):
        if len(self.query)!=5: return
        s0,s1,op_sym,s3,s4=tuple(self.query)
        op_options=[self.op_assign[op_sym]] if op_sym in self.op_assign else list(range(len(OP_NAMES)))
        for d0 in self.vals(s0):
            if not self.assign(s0,d0): continue
            for d1 in self.vals(s1):
                if not self.assign(s1,d1): continue
                left=d0*10+d1
                for d3 in self.vals(s3):
                    if not self.assign(s3,d3): continue
                    for d4 in self.vals(s4):
                        if not self.assign(s4,d4): continue
                        right=d3*10+d4
                        for op_id in op_options:
                            rd=result_digits_for_op(op_id,left,right)
                            if rd is not None:
                                ans=''.join(map(str,rd))
                                self.answers[ans]+=1
                                self.answer_info.setdefault(ans,(dict(self.mapping),dict(self.op_assign)))
                        self.undo(s4)
                    self.undo(s3)
                self.undo(s1)
            self.undo(s0)

def make_d4_trace(prompt, answer, pred, info):
    mapping, op_assign = info
    ops = {k: OP_NAMES[v] for k,v in op_assign.items()} if isinstance(op_assign, dict) else {}
    body = (
        "I solve the symbolic equation rule by fitting the digit symbols and operator symbols to the training examples.\n"
        f"The consistent operator assignments are {ops}. The target evaluates to {pred}."
    )
    return body + "\n</think>\n" + f"\\boxed{{{escape_box_answer(answer)}}}"

d4_rows=[]; d4_stats=Counter()
if not ENABLE_D4:
    print("D4 disabled by default in v61 to avoid the v60 hang. Set ENABLE_D4=True only for capped diagnostics.")
else:
    crypto_df = comp[(comp["family"]=="equation_numeric") & comp["prompt"].astype(str).str.contains(CRYPTO_MARKER, regex=False, na=False)].copy()
    crypto_df = crypto_df.head(CRYPTO_SCAN_MAX_ROWS)
    print("D4 scan rows capped:", len(crypto_df), "timeout_sec:", CRYPTO_ROW_TIMEOUT_SEC)
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    for _,r in crypto_df.iterrows():
        if len(d4_rows) >= CRYPTO_REPAIR_MAX_ROWS: break
        try:
            signal.setitimer(signal.ITIMER_REAL, CRYPTO_ROW_TIMEOUT_SEC)
            parsed = parse_crypto_prompt(str(r["prompt"]))
            pred, info = FastCryptoSolver(parsed).solve()
            signal.setitimer(signal.ITIMER_REAL, 0)
            if pred is None:
                d4_stats["no_unique"] += 1; continue
            if not answer_equal(pred, str(r["answer"])):
                d4_stats["wrong"] += 1; continue
            d4_rows.append({"id":str(r["id"]),"family":"equation_numeric","source":"d4_crypto_timeout_verified",
                            "prompt":str(r["prompt"]),"answer":str(r["answer"]),"assistant":make_d4_trace(str(r["prompt"]),str(r["answer"]),pred,info)})
            d4_stats["verified_exact"] += 1
        except TimeoutException:
            signal.setitimer(signal.ITIMER_REAL, 0); d4_stats["timeout"] += 1
        except Exception:
            signal.setitimer(signal.ITIMER_REAL, 0); d4_stats["error"] += 1
    signal.signal(signal.SIGALRM, old_handler)

print("D4 stats:", d4_stats, "rows:", len(d4_rows))
(OUT_DIR / "v62_d4_rows.jsonl").write_text("\n".join(json.dumps(x,ensure_ascii=False) for x in d4_rows), encoding="utf-8")


In [ ]:

# CELL 6 — Build finisher dataset
train_rows = []
for item in d3_rows:
    for _ in range(D3_UPSAMPLE): train_rows.append(dict(item))
for item in d4_rows:
    for _ in range(D4_UPSAMPLE): train_rows.append(dict(item))

for fam, n in PROTECTIVE_PER_FAMILY.items():
    sub = comp[comp["family"] == fam].copy()
    take = min(n, len(sub))
    sub = sub.sample(n=take, random_state=SEED + len(fam))
    for _, r in sub.iterrows():
        train_rows.append({"id": str(r["id"]), "family": fam, "source": "protective_original_answer_only",
                           "prompt": str(r["prompt"]), "answer": str(r["answer"]), "assistant": answer_only_assistant(r["answer"])})

train_df = pd.DataFrame(train_rows).sample(frac=1.0, random_state=SEED).head(MAX_TRAIN_ROWS).reset_index(drop=True)
for a, ans in zip(train_df["assistant"].astype(str), train_df["answer"].astype(str)):
    boxes = extract_boxed_all(a)
    assert len(boxes) == 1, a[:300]
    assert answer_equal(boxes[-1], ans), (boxes[-1], ans)

print("train_df:", train_df.shape)
print("family counts:"); print(train_df["family"].value_counts())
print("source counts:"); print(train_df["source"].value_counts())
print("sample assistant:"); print(train_df.iloc[0]["assistant"][:1000])
train_df.to_csv(OUT_DIR / "v62_train_df.csv", index=False)
(OUT_DIR / "v62_data_summary.json").write_text(json.dumps({
    "train_shape": list(train_df.shape),
    "family_counts": train_df["family"].value_counts().to_dict(),
    "source_counts": train_df["source"].value_counts().to_dict(),
    "d3_stats": dict(d3_stats),
    "d4_stats": dict(d4_stats),
    "start_adapter": str(START_ADAPTER),
}, indent=2, ensure_ascii=False), encoding="utf-8")


In [ ]:

# CELL 7 — Install Unsloth dependencies and load base + starting adapter
def recursive_wheels(pattern): return sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
def run_cmd(cmd, required=True):
    print("RUN:", " ".join(map(str, cmd)))
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(res.stdout[-2000:])
    if required and res.returncode != 0: raise RuntimeError("Command failed: " + " ".join(map(str, cmd)))
    return res.returncode == 0

import site
PYDEPS = TMP_ROOT / "pydeps"
triton_wheels = sorted(recursive_wheels("triton*.whl"), key=lambda p: (0 if "mayukh18/nemotron-packages/packages" in p else 1, len(p)))
if triton_wheels:
    PYDEPS.mkdir(parents=True, exist_ok=True)
    run_cmd([sys.executable, "-m", "pip", "install", "--no-deps", "--target", str(PYDEPS), "--upgrade", "--ignore-installed", triton_wheels[0]], required=False)
    if str(PYDEPS) not in sys.path: sys.path.insert(0, str(PYDEPS))
    site.addsitedir(str(PYDEPS))

try:
    sys.path.insert(0, "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script")
    ptxas_src = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia_utility_script/triton/backends/nvidia/bin/ptxas-blackwell"
    ptxas_dst = "/tmp/ptxas-blackwell"
    if os.path.exists(ptxas_src) and not os.path.exists(ptxas_dst):
        import stat
        shutil.copy2(ptxas_src, ptxas_dst)
        os.chmod(ptxas_dst, os.stat(ptxas_dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = ptxas_dst
        os.environ["TRITON_PTXAS_PATH"] = ptxas_dst
    import triton.backends.nvidia.compiler as nv_compiler
    nv_compiler.get_ptxas_version = lambda arch: "12.0"
except Exception as e:
    print("Blackwell fix warning:", repr(e))

packages_dir = next((p for p in ["/kaggle/input/datasets/mayukh18/nemotron-packages/packages", "/kaggle/input/mayukh18/nemotron-packages/packages"] if os.path.isdir(p)), None)
if packages_dir is None:
    hits = recursive_wheels("unsloth*.whl"); packages_dir = str(Path(hits[0]).parent) if hits else None
assert packages_dir, "Attach mayukh18/nemotron-packages."
run_cmd([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", packages_dir,
         "unsloth", "trl", "peft", "transformers", "datasets", "accelerate", "bitsandbytes"], required=False)
causal = recursive_wheels("causal*conv1d*.whl"); mamba = recursive_wheels("mamba_ssm-*.whl")
if causal: run_cmd([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", causal[-1]], required=False)
if mamba: run_cmd([sys.executable, "-m", "pip", "install", "--no-index", "--no-deps", mamba[-1]], required=False)

import torch
from unsloth import FastLanguageModel
from safetensors.torch import load_file, save_file
print("Torch", torch.__version__, "CUDA", torch.version.cuda, "available", torch.cuda.is_available())
assert torch.cuda.is_available()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(BASE_MODEL_PATH), max_seq_length=MAX_SEQ_LEN, load_in_4bit=False, load_in_8bit=False,
    full_finetuning=False, trust_remote_code=True, unsloth_force_compile=False, attn_implementation="eager", dtype=torch.bfloat16)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

start_cfg = json.load(open(START_ADAPTER / "adapter_config.json", "r", encoding="utf-8"))
target_modules = list(start_cfg.get("target_modules", []))
r = int(start_cfg.get("r", 32)); alpha = int(start_cfg.get("lora_alpha", start_cfg.get("alpha", 32)))
print("start LoRA config:", {"r": r, "alpha": alpha, "target_modules": target_modules})

model = FastLanguageModel.get_peft_model(model, r=r, lora_alpha=alpha, lora_dropout=0.0,
    target_modules=target_modules, bias="none", use_gradient_checkpointing="unsloth", random_state=SEED)
model.print_trainable_parameters()

start_state = load_file(str(START_ADAPTER / "adapter_model.safetensors"), device="cpu")
current = model.state_dict(); current_keys = set(current.keys())
def add_default_variant(k): return k.replace(".lora_A.weight", ".lora_A.default.weight").replace(".lora_B.weight", ".lora_B.default.weight")
def remove_default_variant(k): return k.replace(".lora_A.default.weight", ".lora_A.weight").replace(".lora_B.default.weight", ".lora_B.weight")
def key_variants(src_k):
    variants = []
    for k in [src_k, add_default_variant(src_k), remove_default_variant(src_k)]:
        variants += [k, k.replace("base_model.model.model.", "base_model.model.backbone."),
                     k.replace("base_model.model.backbone.", "base_model.model.model."),
                     k.replace(".lora_A.", ".lora_A.default.").replace(".lora_B.", ".lora_B.default.")]
    out, seen = [], set()
    for k in variants:
        if k not in seen: out.append(k); seen.add(k)
    return out

src_to_model_key, unmapped = {}, []
for src_k, src_v in start_state.items():
    found = None
    for cand in key_variants(src_k):
        if cand in current_keys and tuple(current[cand].shape) == tuple(src_v.shape):
            found = cand; break
    if found is None: unmapped.append(src_k)
    else: src_to_model_key[src_k] = found
coverage = len(src_to_model_key) / max(1, len(start_state))
print("start tensors", len(start_state), "mapped", len(src_to_model_key), "coverage", coverage, "unmapped first", unmapped[:10])
assert coverage >= 0.995

mapped_for_load = {mk: start_state[sk].to(dtype=current[mk].dtype) for sk, mk in src_to_model_key.items()}
missing, unexpected = model.load_state_dict(mapped_for_load, strict=False)
print("Loaded start adapter. missing:", len(missing), "unexpected:", len(unexpected))
del current, mapped_for_load
gc.collect(); torch.cuda.empty_cache()
show_disk("after model + adapter load")


In [ ]:

# CELL 8 — Short SFT finisher
from torch.utils.data import DataLoader, Sampler
from datasets import Dataset as HFDataset
from trl import SFTTrainer, SFTConfig

records, labels = [], []
for _, r0 in train_df.iterrows():
    records.append(make_chat_target(str(r0["prompt"]), str(r0["assistant"])))
    labels.append(str(r0["family"]))

def formatting_prompts_func(example):
    messages = example["messages"]
    convs = [messages] if messages and isinstance(messages[0], dict) else messages
    texts = []
    for conv in convs:
        try: text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False, enable_thinking=True)
        except TypeError: text = tokenizer.apply_chat_template(conv, tokenize=False, add_generation_prompt=False)
        texts.append(text)
    return texts

def build_order(labels, batch_size, seed):
    by = defaultdict(list)
    for i, lab in enumerate(labels): by[str(lab)].append(i)
    rng = random.Random(seed)
    for v in by.values(): rng.shuffle(v)
    nb = max(1, math.ceil(len(labels) / batch_size)); batches = [[] for _ in range(nb)]
    order = list(range(nb)); rng.shuffle(order); a = 0
    for lab in sorted(by):
        for idx in by[lab]: batches[order[a % nb]].append(idx); a += 1
    return [idx for b in batches for idx in b]

class PrecomputedOrderSampler(Sampler):
    def __init__(self, order): self.order = list(order)
    def __iter__(self): return iter(self.order)
    def __len__(self): return len(self.order)

class StratifiedSFTTrainer(SFTTrainer):
    def __init__(self, *args, stratified_order=None, **kwargs):
        super().__init__(*args, **kwargs); self.stratified_order = stratified_order
    def get_train_dataloader(self):
        if self.stratified_order is None: return super().get_train_dataloader()
        kwargs = {"batch_size": self.args.per_device_train_batch_size, "sampler": PrecomputedOrderSampler(self.stratified_order),
                  "collate_fn": self.data_collator, "num_workers": self.args.dataloader_num_workers,
                  "pin_memory": self.args.dataloader_pin_memory, "persistent_workers": self.args.dataloader_persistent_workers,
                  "drop_last": self.args.dataloader_drop_last}
        if self.args.dataloader_num_workers > 0: kwargs["prefetch_factor"] = self.args.dataloader_prefetch_factor
        return DataLoader(self.train_dataset, **kwargs)

dataset = HFDataset.from_list(records)
effective_batch = PER_DEVICE_BS * GRAD_ACCUM
order = build_order(labels, effective_batch, SEED)
sft_kwargs = dict(
    output_dir=str(TRAIN_DIR), max_steps=MAX_STEPS, per_device_train_batch_size=PER_DEVICE_BS,
    gradient_accumulation_steps=GRAD_ACCUM, learning_rate=LR, lr_scheduler_type="linear",
    warmup_steps=WARMUP_STEPS, max_length=MAX_SEQ_LEN, adam_beta1=0.9, adam_beta2=0.95,
    adam_epsilon=1e-8, weight_decay=0.0, max_grad_norm=1e9, logging_steps=5, save_strategy="no",
    bf16=True, gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_num_workers=2, remove_unused_columns=False, seed=SEED, report_to="none", packing=False, disable_tqdm=True)
if "assistant_only_loss" in inspect.signature(SFTConfig.__init__).parameters:
    sft_kwargs["assistant_only_loss"] = True; print("assistant_only_loss=True enabled")
else:
    print("WARNING: assistant_only_loss not supported.")
args = SFTConfig(**sft_kwargs)
trainer = StratifiedSFTTrainer(model=model, args=args, train_dataset=dataset, processing_class=tokenizer,
                               formatting_func=formatting_prompts_func, stratified_order=order)
print("Train rows:", len(records), "Effective batch:", effective_batch, "MAX_STEPS:", MAX_STEPS, "LR:", LR)
print("Label counts:", Counter(labels))
show_disk("before v62 finisher training")
t0 = time.time()
train_result = trainer.train()
elapsed_min = (time.time() - t0) / 60
print("train_result:", train_result, "elapsed_min:", round(elapsed_min, 2))
try:
    del trainer, dataset
    gc.collect(); torch.cuda.empty_cache()
except Exception: pass
if TRAIN_DIR.exists(): shutil.rmtree(TRAIN_DIR, ignore_errors=True)
show_disk("after v62 finisher training")


In [ ]:
# CELL 9 — Sparse-trust blend and package
# v61 applied ~8351 tensors. v62 computes the full finisher delta but applies only
# the strongest stable tensor slice, capped at MAX_SELECTED_TENSORS.

def trained_key_variants(src_k):
    variants = []
    for k in key_variants(src_k): variants += [k, remove_default_variant(k), add_default_variant(k)]
    out, seen = [], set()
    for k in variants:
        if k not in seen: out.append(k); seen.add(k)
    return out

def tensor_layer(k):
    m = re.search(r"layers\.(\d+)\.", k)
    return int(m.group(1)) if m else -1

def is_expert_tensor(k):
    return "experts" in k

def is_late_expert_tensor(k):
    if not SKIP_LATE_EXPERTS or "experts" not in k: return False
    layer = tensor_layer(k)
    return layer >= LATE_EXPERT_LAYER_START if layer >= 0 else False

def tensor_kind(k):
    if "lm_head" in k: return "lm_head"
    if "experts" in k: return "expert"
    if any(x in k for x in ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj", "out_proj", "down_proj", "up_proj"]): return "projection"
    return "other"

if TRAINED_ADAPTER_DIR.exists(): shutil.rmtree(TRAINED_ADAPTER_DIR, ignore_errors=True)
TRAINED_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(TRAINED_ADAPTER_DIR))
tokenizer.save_pretrained(str(TRAINED_ADAPTER_DIR / "tokenizer_snapshot"))

trained_state = load_file(str(TRAINED_ADAPTER_DIR / "adapter_model.safetensors"), device="cpu")
trained_keys = set(trained_state.keys())
print("trained tensors:", len(trained_state), "start tensors:", len(start_state))

# Pass 1: scan candidate tensors.
candidates, rel_rows = [], []
scan_stats = Counter()
for src_k, base_v in start_state.items():
    found = None
    for cand in trained_key_variants(src_k):
        if cand in trained_keys and tuple(trained_state[cand].shape) == tuple(base_v.shape):
            found = cand; break
    if found is None:
        scan_stats["missing_tensor"] += 1; continue
    scan_stats["mapped"] += 1
    if PRESERVE_LM_HEAD and "lm_head" in src_k:
        scan_stats["skip_lm_head"] += 1; continue
    if is_late_expert_tensor(src_k):
        scan_stats["skip_late_expert"] += 1; continue

    tr_v = trained_state[found].to(dtype=base_v.dtype)
    delta = tr_v.float() - base_v.float()
    rel = float(delta.norm() / (base_v.float().norm() + 1e-8)) if base_v.numel() else 0.0
    absmax = float(delta.abs().max()) if delta.numel() else 0.0
    kind = tensor_kind(src_k)
    layer = tensor_layer(src_k)

    row = {"tensor": src_k, "rel_delta": rel, "absmax_delta": absmax, "kind": kind, "layer": layer}
    if len(rel_rows) < 12000: rel_rows.append(row)

    if absmax > MAX_ABSMAX_DELTA:
        scan_stats["skip_abs"] += 1; continue
    if rel > MAX_REL_DELTA:
        scan_stats["skip_rel"] += 1; continue
    if rel < NONEXPERT_MIN_REL_DELTA:
        scan_stats["skip_tiny"] += 1; continue
    if kind == "expert":
        if layer >= EXPERT_LAYER_HARD_CAP:
            scan_stats["skip_expert_layer_cap"] += 1; continue
        if rel > EXPERT_MAX_REL_DELTA:
            scan_stats["skip_expert_rel"] += 1; continue

    # Stability score: prefer stronger but still-trusted deltas; penalize high absmax and later experts.
    rel_score = rel / max(MAX_REL_DELTA, 1e-12)
    abs_score = max(0.0, 1.0 - absmax / max(MAX_ABSMAX_DELTA, 1e-12))
    kind_bonus = 0.10 if kind != "expert" else 0.0
    layer_penalty = 0.0
    if kind == "expert" and layer >= 0:
        layer_penalty = 0.15 * (layer / max(1, EXPERT_LAYER_HARD_CAP))
    score = float(rel_score + 0.25 * abs_score + kind_bonus - layer_penalty)
    candidates.append({"tensor": src_k, "trained_key": found, "rel_delta": rel, "absmax_delta": absmax, "kind": kind, "layer": layer, "score": score})
    scan_stats["candidate"] += 1

candidates = sorted(candidates, key=lambda x: x["score"], reverse=True)
expert_cap = int(MAX_SELECTED_TENSORS * EXPERT_MAX_FRACTION)
selected, expert_selected = [], 0
for c in candidates:
    if len(selected) >= MAX_SELECTED_TENSORS: break
    if c["kind"] == "expert":
        if expert_selected >= expert_cap: continue
        expert_selected += 1
    selected.append(c)
selected_set = {c["tensor"] for c in selected}
print("Sparse scan stats:", dict(scan_stats))
print("candidate_count:", len(candidates), "selected:", len(selected), "expert_selected:", expert_selected, "expert_cap:", expert_cap)
assert len(selected) >= MIN_SELECTED_TENSORS, f"Too few sparse tensors selected: {len(selected)} < {MIN_SELECTED_TENSORS}"

# Pass 2: build blended state; all non-selected tensors remain exactly at start adapter.
blended_state = {}
applied = kept_start = missing_tensor = 0
max_rel = 0.0; max_abs = 0.0
selected_lookup = {c["tensor"]: c for c in selected}
for src_k, base_v in start_state.items():
    c = selected_lookup.get(src_k)
    if c is None:
        blended_state[src_k] = base_v; kept_start += 1; continue
    found = c["trained_key"]
    tr_v = trained_state[found].to(dtype=base_v.dtype)
    delta = tr_v.float() - base_v.float()
    rel = float(delta.norm() / (base_v.float().norm() + 1e-8)) if base_v.numel() else 0.0
    absmax = float(delta.abs().max()) if delta.numel() else 0.0
    max_rel = max(max_rel, rel); max_abs = max(max_abs, absmax)
    blended_state[src_k] = (base_v.float() + float(BLEND_ALPHA) * delta).to(base_v.dtype)
    applied += 1

coverage = scan_stats["mapped"] / max(1, len(start_state))
stats = {"mapped": int(scan_stats["mapped"]), "coverage": coverage, "candidate_count": len(candidates),
         "selected": len(selected), "applied": applied, "kept_start": kept_start,
         "expert_selected": expert_selected, "expert_cap": expert_cap,
         "scan_stats": dict(scan_stats), "max_rel_selected": max_rel, "max_abs_selected": max_abs,
         "blend_alpha": BLEND_ALPHA, "max_selected_tensors": MAX_SELECTED_TENSORS,
         "expert_layer_hard_cap": EXPERT_LAYER_HARD_CAP, "expert_max_rel_delta": EXPERT_MAX_REL_DELTA}
print(json.dumps(stats, indent=2)[:6000])
assert coverage >= 0.995
assert applied == len(selected)
assert applied >= REQUIRE_MIN_APPLIED

if BLENDED_ADAPTER_DIR.exists(): shutil.rmtree(BLENDED_ADAPTER_DIR, ignore_errors=True)
BLENDED_ADAPTER_DIR.mkdir(parents=True, exist_ok=True)
cfg = patch_config_for_submission(json.load(open(START_ADAPTER / "adapter_config.json", "r", encoding="utf-8")))
cfg["v62_d3_sparse_trust_note"] = {
    "method": "D3 substitution-cipher verified finisher with protective replay; sparse-trust merge into starting adapter",
    "start_adapter": str(START_ADAPTER),
    "d3_verified_rows": int(len(d3_rows)), "d4_verified_rows": int(len(d4_rows)),
    "train_rows": int(len(train_df)), "family_counts": train_df["family"].value_counts().to_dict(),
    "source_counts": train_df["source"].value_counts().to_dict(),
    "max_steps": MAX_STEPS, "lr": LR, "blend_alpha": BLEND_ALPHA,
    "max_rel_delta": MAX_REL_DELTA, "max_absmax_delta": MAX_ABSMAX_DELTA,
    "max_selected_tensors": MAX_SELECTED_TENSORS, "min_selected_tensors": MIN_SELECTED_TENSORS,
    "preserve_lm_head": PRESERVE_LM_HEAD, "skip_late_experts": SKIP_LATE_EXPERTS,
    "late_expert_layer_start": LATE_EXPERT_LAYER_START,
    "expert_layer_hard_cap": EXPERT_LAYER_HARD_CAP, "expert_max_rel_delta": EXPERT_MAX_REL_DELTA,
    "blend_stats": stats,
}
(BLENDED_ADAPTER_DIR / "adapter_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
save_file(blended_state, str(BLENDED_ADAPTER_DIR / "adapter_model.safetensors"))

pd.DataFrame(rel_rows).sort_values("rel_delta", ascending=False).to_csv(OUT_DIR / "v62_delta_scan_all.csv", index=False)
pd.DataFrame(selected).to_csv(OUT_DIR / "v62_selected_sparse_tensors.csv", index=False)

# Only now create the real submission.zip.
if SUBMISSION_ZIP.exists(): SUBMISSION_ZIP.unlink()
final_tmp_zip = TMP_ROOT / "submission_v62_candidate.zip"
package_adapter_to_zip(BLENDED_ADAPTER_DIR, final_tmp_zip)
shutil.move(str(final_tmp_zip), str(SUBMISSION_ZIP))
validate_submission_zip(SUBMISSION_ZIP)

summary = {"version": "v62_d3_sparse_trust_finisher",
           "selected_final": "v62_d3_sparse_trust_finisher_candidate",
           "start_adapter": str(START_ADAPTER), "submission_zip": str(SUBMISSION_ZIP),
           "submission_size_gb": SUBMISSION_ZIP.stat().st_size / 1024**3,
           "d3_verified_rows": int(len(d3_rows)), "d4_verified_rows": int(len(d4_rows)),
           "train_rows": int(len(train_df)), "family_counts": train_df["family"].value_counts().to_dict(),
           "source_counts": train_df["source"].value_counts().to_dict(), "train_elapsed_min": float(elapsed_min),
           "blend_alpha": BLEND_ALPHA, "blend_stats": stats}
(OUT_DIR / "v62_final_summary.json").write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2)[:10000])
print("READY TO SUBMIT:", SUBMISSION_ZIP, "sizeGB", validate_submission_zip(SUBMISSION_ZIP), "FINAL v62_d3_sparse_trust_finisher_candidate")
show_disk("final")


In [ ]:

# CELL 10 — Reports archive
REPORT_ARCHIVE = WORK / "v62_reports_and_logs.zip"
REPORT_ARCHIVE.unlink(missing_ok=True)
with zipfile.ZipFile(REPORT_ARCHIVE, "w", zipfile.ZIP_DEFLATED, allowZip64=True) as z:
    if OUT_DIR.exists():
        for p in OUT_DIR.rglob("*"):
            if p.is_file():
                z.write(p, p.relative_to(WORK))
print("Important outputs:")
print(" - Final submission:", SUBMISSION_ZIP, "valid_size_gb", validate_submission_zip(SUBMISSION_ZIP))
print(" - Reports archive:", REPORT_ARCHIVE, "sizeMB", round(REPORT_ARCHIVE.stat().st_size / 1024**2, 3))
print(" - Summary:", OUT_DIR / "v62_final_summary.json")
print("Submit manually only if final said FINAL v62_d3_sparse_trust_finisher_candidate")
